# 1. Import libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

import numpy as np
from sklearn.metrics import balanced_accuracy_score

from sklearn.model_selection import GroupKFold
import torch
import gc

In [ ]:
DATA = 'data/'

# 2. Load dataset

In [ ]:
EXTRA = f'{DATA}/nlm-matched-clean/'

extra_data = pd.read_parquet(f'{EXTRA}/nlm_pool_v5.parquet')   
labels = sorted(extra_data['model'].unique())       
label_to_index = {m: i for i, m in enumerate(labels)}
extra_data['model_id'] = extra_data['model'].map(label_to_index).to_numpy()

print(label_to_index)
extra_data

In [ ]:
model_counts = extra_data.pivot_table(
    index='prompt_key', 
    columns='model', 
    aggfunc='size', 
    fill_value=0
).reset_index()

model_counts

In [ ]:
model_columns = model_counts.columns.drop('prompt_key')
model_counts_cleaned = model_counts[(model_counts[model_columns] != 0).all(axis=1)]
model_counts_cleaned

In [ ]:
# take ~10% of passages from the pool of prompts that all 18 models answered
rng = np.random.default_rng(42)
n_hold = int(round(0.1 * len(model_counts_cleaned)))
print("Number of held out prompts:", n_hold)

hold_prompts = rng.choice(model_counts_cleaned['prompt_key'], size=n_hold, replace=False)

is_hold = extra_data['prompt_key'].isin(hold_prompts)
stage1_train_df = extra_data[~is_hold].reset_index(drop=True)
stage1_val_df  = extra_data[is_hold].reset_index(drop=True)

print(f"Length of stage 1 train data: {len(stage1_train_df)}")
print(f"Length of stage 1 validation data: {len(stage1_val_df)}")

# 3. Stage 1: Fine-tune the encoder on extra data

## 3.1 Convert data into required format

In [ ]:
stage1_train_dataset = Dataset.from_pandas(
    stage1_train_df[['text', 'model_id']].rename(columns={'model_id': 'labels'}),
    preserve_index=False
)

In [ ]:
stage1_val_dataset = Dataset.from_pandas(
    stage1_val_df[['text', 'model_id']].rename(columns={'model_id': 'labels'}),
    preserve_index=False
)

## 3.2 Tokenize dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=384)
 
tokenized_stage1_train = stage1_train_dataset.map(tokenize, batched=True, remove_columns=['text'])
tokenized_stage1_train.features.keys() 

In [ ]:
tokenized_stage1_val = stage1_val_dataset.map(tokenize, batched=True, remove_columns=['text'])

In [ ]:
id_to_label = {i: m for i, m in enumerate(labels)}
label_to_id = {m: i for i, m in id_to_label.items()}

## 3.3 Creating model and training arguments

steps = passages ÷ (batch size × accumulation steps)

In [ ]:
def compute_metrics(p):
    return {"balanced_acc": balanced_accuracy_score(p.label_ids, p.predictions.argmax(-1))}

In [ ]:
model_stage1 = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base", 
    num_labels=len(labels), 
    label2id=label_to_id, 
    id2label=id_to_label,
)

training_args_stage_1 = TrainingArguments(
    output_dir="ModernBERT-stage-1",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=64,
    learning_rate=1e-5,
    num_train_epochs=10,            
    logging_steps=500,
    eval_strategy="steps",         
    eval_steps=500,                
    save_strategy="steps",
    save_steps=500,                      # must match eval_steps
    save_total_limit=2,                   # keeps the best and the latest checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="balanced_acc",
    greater_is_better=True,
    fp16=False, # True on Nvidia GPU
    bf16=True, # True on mac 
    report_to="none",
)

trainer_stage1 = Trainer(
    model=model_stage1,
    args=training_args_stage_1,
    train_dataset=tokenized_stage1_train,
    eval_dataset=tokenized_stage1_val,        
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_stage1.train()
trainer_stage1.save_model("stage1_modernbert")   

# 4. Stage 2: Fine-tune stage 1 weights on official data

## 4.1 Load official dataset

In [ ]:
official_data = pd.read_csv(f'{DATA}/train.csv')

official_data['model_id'] = official_data['model'].map(label_to_index).to_numpy()
official_data

## 4.2 Convert data into required format

In [ ]:
official_dataset = Dataset.from_pandas(
    official_data[['text', 'model_id']].rename(columns={'model_id': 'labels'}),
    preserve_index=False
)

## 4.3 Tokenize dataset

In [ ]:
fine_tuned_tokenizer = AutoTokenizer.from_pretrained("stage1_modernbert")

def tokenize(batch):
    return fine_tuned_tokenizer(batch['text'], truncation=True, max_length=384)

tokenized_official_data = official_dataset.map(tokenize, batched=True, remove_columns=['text'])

In [ ]:
def softmax(logits):
    return torch.softmax(torch.tensor(logits), dim=-1).numpy()

all_preds = []

for fold, (train_idx, val_idx) in enumerate(
    GroupKFold(n_splits=5).split(official_data, groups=official_data['prompt_id'])
):
    val_df = official_data.iloc[val_idx].copy()
    tokenized_stage2_train = tokenized_official_data.select(train_idx)
    tokenized_stage2_val = tokenized_official_data.select(val_idx)

    model_stage2 = AutoModelForSequenceClassification.from_pretrained("stage1_modernbert") 

    training_args_stage_2 = TrainingArguments(
        output_dir=f"ModernBERT-stage-2-fold-{fold}",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        per_device_eval_batch_size=64,
        learning_rate=1e-5,
        num_train_epochs=10,
        logging_steps=50,
        eval_strategy="epoch",       # score on the held-out prompts after every epoch
        save_strategy="no",
        fp16=False, # True on Nvidia GPU
        bf16=True, # True on mac 
        report_to="none",
    )

    trainer_stage2 = Trainer(
        model=model_stage2,
        args=training_args_stage_2,
        train_dataset=tokenized_stage2_train,
        eval_dataset=tokenized_stage2_val,
        processing_class=fine_tuned_tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer_stage2.train()
    probs = softmax(trainer_stage2.predict(tokenized_stage2_val).predictions)

    val_df['pred'] = [labels[i] for i in probs.argmax(1)]
    val_df['fold'] = fold
    all_preds.append(val_df)

    del trainer_stage2, model_stage2
    gc.collect()
    torch.mps.empty_cache()

predictions_df = pd.concat(all_preds)
predictions_df

In [ ]:
score = balanced_accuracy_score(predictions_df['model'], predictions_df['pred'])
print(f'Balanced accuracy = {score:.3f}')

# recall per model
predictions_df['correct'] = predictions_df['model'] == predictions_df['pred']

recall_df = (predictions_df.groupby('model')['correct']
                           .mean()
                           .round(2)
                           .sort_values(ascending=False)
                           .reset_index(name='recall'))
recall_df